# 03 · Feature Engineering

> **Objetivo:** convertir un dataset *transaccional* en un dataset *por cliente*,
> donde cada fila representa el comportamiento agregado de un cliente.

## ¿Qué vamos a hacer?

1. Definir el **snapshot date** (fecha de referencia).
2. Calcular RFM clásico: **Recency**, **Frequency**, **Monetary**.
3. Extender RFM con: **AvgTicket**, **ProductDiversity**, **AvgQuantity**.
4. Validar la calidad de las features.
5. Guardar el dataset por cliente.

## Concepto teórico: ¿por qué RFM?

RFM nació en los 90s en marketing directo. La intuición:

- *Si compraste recientemente, es más probable que vuelvas a comprar.*
- *Si compras seguido, ya eres un cliente fiel.*
- *Si gastas mucho, eres valioso para el negocio.*

Décadas de evidencia empírica respaldan que estas tres dimensiones,
incluso sin más features, ya capturan la mayor parte de la variabilidad
de comportamiento de clientes en retail.

> **Tip didáctico:** RFM es el "Hello World" de la segmentación.
> Empieza siempre por aquí antes de meter features más sofisticadas.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd
import numpy as np

from src.features.rfm import build_customer_features, build_rfm
from src.config import CLEAN_DATA_FILE, RFM_DATA_FILE, FEATURES_DATA_FILE


## 1. Cargar las transacciones limpias

In [ ]:
df = pd.read_parquet(CLEAN_DATA_FILE)
print(f"Transacciones: {len(df):,}")
print(f"Clientes únicos: {df['CustomerID'].nunique():,}")


## 2. Snapshot date

Es la fecha "hoy" desde la cual medimos `Recency`. Usamos `max + 1 día`
para evitar que algún cliente tenga `Recency = 0`.


In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")


## 3. RFM clásico

Calculamos las tres métricas con un solo `groupby`:


In [ ]:
rfm = build_rfm(df, snapshot_date=snapshot_date)
rfm.head()


**Lectura del output:**

- `Recency` en días.
- `Frequency` = nº de facturas distintas.
- `Monetary` = gasto total acumulado.

Un cliente con `Recency=2, Frequency=10, Monetary=5000` es muy valioso.
Uno con `Recency=300, Frequency=1, Monetary=20` es probablemente perdido.

## 4. Distribución de RFM


In [ ]:
rfm.describe().T


> **Observa el sesgo:** la mediana de `Monetary` es mucho menor que la
> media. Eso confirma una distribución long-tail (pocos clientes muy
> grandes inflan la media). Tendremos que aplicar `log1p` antes del
> escalado.

## 5. Features extendidas

Tres variables adicionales que ayudan al clustering a distinguir matices:

- **AvgTicket** = `Monetary / Frequency`. Distingue clientes de muchas
  compras pequeñas vs. pocas compras grandes.
- **ProductDiversity** = nº de StockCode distintos. Un cliente con
  diversidad 1 compra siempre lo mismo (¿suministro?, ¿hábito?).
- **AvgQuantity** = cantidad promedio de items por factura. Alta
  podría indicar comportamiento mayorista.


In [ ]:
features = build_customer_features(df, snapshot_date=snapshot_date)
features.head()


## 6. Estadísticas y correlaciones

In [ ]:
features.describe().T


In [ ]:
features.corr().round(2)


> **Esperado:** `Frequency`, `Monetary` y `ProductDiversity` están
> fuertemente correlacionadas. Tiene sentido: comprar más veces implica
> gastar más y probar más productos. La correlación es información, no
> un problema. Lo importante es que K-Means y DBSCAN no requieren
> independencia (a diferencia de la regresión).

## 7. Validación de calidad


In [ ]:
checks = {
    "Filas con NaN": features.isna().any(axis=1).sum(),
    "Recency negativa": (features["Recency"] < 0).sum(),
    "Frequency cero": (features["Frequency"] <= 0).sum(),
    "Monetary cero o negativa": (features["Monetary"] <= 0).sum(),
}
for k, v in checks.items():
    status = "OK" if v == 0 else "REVISAR"
    print(f"  [{status}] {k}: {v}")


## 8. Top 10 clientes por valor


In [ ]:
features.nlargest(10, "Monetary")


> Estos clientes son los Champions / VIPs. Es probable que terminen en
> un cluster aparte (o sean marcados como outliers por DBSCAN).

## 9. Guardar features


In [ ]:
rfm.to_parquet(RFM_DATA_FILE)
features.to_parquet(FEATURES_DATA_FILE)
print(f"RFM básico:    {RFM_DATA_FILE}")
print(f"Features ext.: {FEATURES_DATA_FILE}")


## Resumen

| Feature | Calculo | Insight de negocio |
|---|---|---|
| Recency | Días desde última compra | Bajo = activo |
| Frequency | Nº de facturas | Alto = fiel |
| Monetary | Gasto total | Alto = valioso |
| AvgTicket | Monetary / Frequency | Distingue volumen vs. ticket |
| ProductDiversity | Nº productos únicos | Alto = explorador |
| AvgQuantity | Items promedio por factura | Alto = mayorista? |

---

## Preguntas de Reflexión

1. ¿Por qué `Frequency` y `Monetary` tienen correlación alta? ¿Eso
   significa que una de las dos es redundante? Justifica.
2. Si calcularas `Recency` con `snapshot_date = '2010-01-01'` (en medio
   del dataset), ¿qué problemas tendrías?
3. ¿Qué otra feature de comportamiento se te ocurre que no esté aquí?
4. Para un negocio **B2B** (suministros industriales), ¿modificarías
   alguna definición de RFM?

> **Próximo paso:** ``04_exploratory_data_analysis.ipynb`` — visualizar
> las features para tomar decisiones de modelado.
